# LSD1 structure-based screen

Ligand-based screening came back empty across 1.1M compounds from ChEMBL and
ZINC: the applicability domain rejected 91% of catalogue chemistry, and nothing
scored above p_active 0.25. That result stands - novel LSD1 chemotypes don't
resemble the known ones closely enough for a similarity-driven model to find
them.

Docking has no such limitation: it scores against the protein structure, so
chemistry unlike anything in ChEMBL is fair game.

**Files:** `actives.csv` (smiles, pchembl) and `library.csv` (smiles, catalog_id).
The first cell puts them on Google Drive, so a runtime restart no longer costs
you the uploads or the docking checkpoints.

**Run the cells in order, top to bottom.** Each one defines things the next
needs. A cell run out of order fails with a NameError, or worse, sits waiting
on a Drive authorization popup you did not notice.

Cell 4 is a one-second self-check. Run it any time something looks stuck: it
distinguishes "a cell above did not finish" from "this is genuinely slow".

Timing on 2 CPUs is the usual surprise. Docking is roughly 20-60 seconds per
compound there, so validation (20 docks) is 10-20 minutes of mostly silent
work. The preflight cell measures this first so you know what to expect.

Runtime -> Change runtime type -> more CPUs. Vina is CPU-bound, GPU is unused.

In [ ]:
# ---- persistent working directory ------------------------------------------
# Colab wipes /content on every runtime restart. Google Drive survives, so we
# work directly on it: after os.chdir every read and write lands on Drive,
# including the docking checkpoints.
from google.colab import drive, files
import os

drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/lsd1"
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)

need = [f for f in ("actives.csv", "library.csv") if not os.path.exists(f)]
if need:
    print("missing:", need, "-> upload now, this is the last time")
    up = files.upload()
    for name, data in up.items():
        clean = name.replace(" (1)", "").replace(" (2)", "")   # Colab suffixes dupes
        open(clean, "wb").write(data)
        print("  saved", clean)

print("\ncwd:", os.getcwd())
print("csv:", [f for f in os.listdir() if f.endswith(".csv")])

## 1. Environment

The `vina` PyPI package compiles from source against Boost and fails on recent
Python versions. The official release binary needs no compilation.

In [ ]:
# no compilation: the vina PyPI package builds from source and fails on recent
# Python. The official release binary just runs.
!pip install -q rdkit meeko gemmi
!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.7/vina_1.2.7_linux_x86_64 -O /usr/local/bin/vina
!chmod +x /usr/local/bin/vina
!vina --version

import os, re, glob, time, shutil, subprocess, tempfile
import numpy as np, pandas as pd, gemmi
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import AllChem, Descriptors, Crippen, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from meeko import MoleculePreparation, PDBQTWriterLegacy

RDLogger.DisableLog("rdApp.*")
FPGEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# Structures WITH a bound reversible inhibitor. The crystallographer already
# located the pocket, which beats any geometric heuristic for placing the box.
# 2V1D was the wrong choice: it contains only FAD, so there was nothing to
# anchor on and the guessed centre landed inside solid protein.
PDB_CANDIDATES = ["5LHH", "9DBP", "5LGT", "6TUZ"]
PDB_ID  = PDB_CANDIDATES[0]
EXHAUST = 4         # 4 to triage on few CPUs; 8-16 once it validates
CENTRE_OVERRIDE = None   # e.g. [-6.5, 66.7, 86.0] to place the box by hand
BOX     = 22.0      # angstroms, cubic
N_DOCK  = 300       # one sitting at ~30s/compound on 2 CPUs
AFFINITY_CUTOFF = None   # set by validation; None = not calibrated yet
TIME_BUDGET_MIN = 180  # stop cleanly instead of running until Colab drops you
NCPU    = os.cpu_count()
print(f"\n{NCPU} CPUs available")

## 2. Receptor

**Two failed runs taught this cell what it now does.** `remove_ligands_and_waters()`
deleted FAD along with the waters. FAD is a covalently attached cofactor, not a
ligand: strip it and you open a 53-atom cavity that Vina fills happily with
anything, which is why random ZINC compounds outscored real nanomolar
inhibitors by 1.2 kcal/mol.

v6 then guessed the pocket from FAD geometry and put the box inside solid
protein: every score came back positive, which means steric clash rather than
binding.

So this version stops guessing. It tries structures known to contain a bound
reversible inhibitor, takes the first one that actually has a druglike het, and
centres the box on that ligand before removing it. The crystallographer located
the pocket; use their answer.

In [ ]:
# ---- receptor ---------------------------------------------------------------
# Two rules learned the hard way:
#   1. FAD is a COFACTOR, not a ligand. Strip it and you open a 53-atom cavity
#      that Vina fills happily with anything -> random compounds outscore real
#      inhibitors (v5 failed this way, separation -1.2 kcal/mol).
#   2. Do not guess the pocket from cofactor geometry. An offset from FAD N5
#      put the box inside solid protein -> every score came back POSITIVE,
#      meaning steric clash, not binding (v6 failed this way, +49 vs +17).
# The fix for both: use a structure with a bound inhibitor and centre on it.
KEEP_HET = {"FAD", "FMN", "NAD", "NAP", "SAM", "HEM",
            "ZN", "MG", "MN", "FE", "CA", "CU"}
JUNK_HET = {"HOH", "SO4", "GOL", "EDO", "PO4", "ACT", "PEG", "DMS", "TRS",
            "MES", "CL", "NA", "K", "IOD", "BME", "MPD", "FMT", "CIT", "TLA"}

def inventory(pdb_id):
    """Download a structure and report its heteroatom groups."""
    if not os.path.exists(f"{pdb_id}.cif"):
        os.system(f"wget -q https://files.rcsb.org/download/{pdb_id}.cif")
    if not os.path.exists(f"{pdb_id}.cif") or os.path.getsize(f"{pdb_id}.cif") < 1000:
        return None, {}
    st = gemmi.read_structure(f"{pdb_id}.cif")
    st.setup_entities()
    hets = {}
    for model in st:
        for ch in model:
            for res in ch:
                if res.het_flag == "H" and res.name not in JUNK_HET:
                    hets.setdefault(res.name, []).append(
                        {a.name: np.array([a.pos.x, a.pos.y, a.pos.z]) for a in res})
    return st, hets

# pick the first candidate that actually has a druglike bound ligand
chosen = None
for pid in ([PDB_ID] if CENTRE_OVERRIDE is not None else PDB_CANDIDATES):
    st, hets = inventory(pid)
    if st is None:
        print(f"  {pid}: download failed")
        continue
    ligs = {n: h for n, h in hets.items()
            if n not in KEEP_HET and len(h[0]) >= 15}      # >=15 atoms = druglike
    summary = ", ".join(f"{n}({len(h[0])} atoms)" for n, h in hets.items()) or "none"
    print(f"  {pid}: {summary}")
    if ligs:
        chosen = (pid, st, hets, ligs)
        break

if chosen is None:
    raise SystemExit("No candidate had a bound ligand. Set CENTRE_OVERRIDE by hand "
                     "after inspecting the structure, or add more PDB_CANDIDATES.")

PDB_ID, st, hets, ligs = chosen
print(f"\nusing {PDB_ID}")

# --- box centre = the bound inhibitor -----------------------------------------
if CENTRE_OVERRIDE is not None:
    centre = np.array(CENTRE_OVERRIDE, float)
    print(f"box centre: manual override {centre.round(2)}")
else:
    LIG = max(ligs, key=lambda n: len(ligs[n][0]))
    coords = np.array(list(ligs[LIG][0].values()))
    centre = coords.mean(axis=0)
    span = coords.max(axis=0) - coords.min(axis=0)
    print(f"box centre: bound inhibitor {LIG} at {centre.round(2)}")
    print(f"  ligand spans {span.round(1)} A; BOX={BOX} must comfortably exceed that")
    if span.max() + 8 > BOX:
        print(f"  WARNING: BOX may be too small. Consider BOX = {int(span.max()+10)}")

# --- strip: protein + cofactors only -----------------------------------------
st.remove_hydrogens()
st.remove_alternative_conformations()
st.write_pdb("receptor_raw.pdb")

kept = []
with open("receptor_raw.pdb") as fh, open("receptor.pdb", "w") as out:
    for line in fh:
        if line.startswith("HETATM"):
            rn = line[17:20].strip()
            if rn in KEEP_HET:
                out.write(line); kept.append(rn)
            continue                      # the bound inhibitor goes too - it is
        if line.startswith(("ATOM", "TER", "END")):   # what we are redocking
            out.write(line)
print(f"\ncofactors retained: {sorted(set(kept)) or 'NONE'}")

# --- PDBQT --------------------------------------------------------------------
ok = False
try:
    from meeko import PDBQTReceptor
    PDBQTReceptor.from_pdb_file("receptor.pdb").write_pdbqt_file("receptor.pdbqt")
    ok = True
except Exception as e:
    print(f"meeko receptor path unavailable ({type(e).__name__}), using openbabel")
if not ok:
    os.system("apt-get -qq install -y openbabel > /dev/null 2>&1")
    os.system("obabel receptor.pdb -O receptor.pdbqt -xr --partialcharge gasteiger 2>/dev/null")

sz = os.path.getsize("receptor.pdbqt") if os.path.exists("receptor.pdbqt") else 0
assert sz > 10_000, "receptor prep failed - stop, everything downstream would be noise"
n_cof = sum(1 for l in open("receptor.pdbqt")
            if l.startswith(("ATOM", "HETATM")) and l[17:20].strip() in KEEP_HET)
print(f"receptor.pdbqt: {sz//1024} KB, {n_cof} cofactor atoms included")
if n_cof == 0 and any(n in KEEP_HET for n in hets):
    print("  WARNING: cofactor lost in PDBQT conversion. The cavity is open again.")

## 3. Ligand prep and the docking call

The `vina` PyPI package compiles from source against Boost and fails on recent
Python versions, which is what broke the previous notebook. The official release
binary needs no compilation, so this calls it through subprocess instead.

The smoke test at the bottom tells you the per-compound cost, which is how you
size `N_DOCK`.

In [ ]:
# ---- ligand prep + docking wrapper -----------------------------------------
def to_pdbqt(smiles, seed=0xf00d):
    """SMILES -> 3D -> PDBQT string. Vina searches torsions itself, so one
    ETKDG conformer is enough."""
    m = Chem.MolFromSmiles(smiles)
    if m is None:
        return None
    m = Chem.AddHs(m)
    # maxAttempts is critical: the default (0) means unlimited, and embedding
    # can spin for many minutes on awkward structures. That is the only place
    # in this pipeline that can hang without a timeout.
    if AllChem.EmbedMolecule(m, maxAttempts=20, randomSeed=seed,
                             useRandomCoords=True) != 0:
        return None
    try:
        AllChem.MMFFOptimizeMolecule(m, maxIters=400)
    except Exception:
        pass
    try:
        s, good, _ = PDBQTWriterLegacy.write_string(MoleculePreparation().prepare(m)[0])
        return s if good else None
    except Exception:
        return None


def dock_one(pdbqt_str, centre=None, box=None, exhaust=None, cpu=None, timeout=300):
    """Best affinity in kcal/mol (lower = better), or None."""
    centre = centre if centre is not None else globals()["centre"]
    box, exhaust = box or BOX, exhaust or EXHAUST
    with tempfile.TemporaryDirectory() as d:
        lig = f"{d}/lig.pdbqt"
        open(lig, "w").write(pdbqt_str)
        cmd = ["vina", "--receptor", "receptor.pdbqt", "--ligand", lig,
               "--out", f"{d}/out.pdbqt",
               "--center_x", f"{centre[0]:.3f}", "--center_y", f"{centre[1]:.3f}",
               "--center_z", f"{centre[2]:.3f}",
               "--size_x", str(box), "--size_y", str(box), "--size_z", str(box),
               "--exhaustiveness", str(exhaust), "--cpu", str(cpu or NCPU),
               "--seed", "42"]
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        except subprocess.TimeoutExpired:
            return None
        m = re.search(r"^\s*1\s+(-?\d+\.\d+)", r.stdout, re.M)
        return float(m.group(1)) if m else None


# smoke test on tranylcypromine, the classic LSD1 chemotype
pq = to_pdbqt("NC1CC1c1ccccc1")
print("ligand prep:", "OK" if pq else "FAILED")
t = time.time(); a = dock_one(pq); dt = time.time() - t
print(f"dock: {a} kcal/mol in {dt:.1f}s  ->  ~{dt*N_DOCK/60:.0f} min for {N_DOCK} compounds")

## 4. Self-check

One second to run. If a later cell seems to hang, run this first: it tells you
whether the pieces it depends on actually exist. Every "MISS" means a cell
above did not finish.

In [ ]:
# ---- self-check: is everything the later cells need actually here? ---------
# Run this any time something seems stuck. It never takes more than a second.
import os, shutil

def _state():
    yield "vina binary", shutil.which("vina") is not None
    yield "receptor.pdbqt", (os.path.exists("receptor.pdbqt") and
                             os.path.getsize("receptor.pdbqt") > 10_000)
    yield "pocket centre", "centre" in globals()
    yield "to_pdbqt()", "to_pdbqt" in globals()
    yield "dock_one()", "dock_one" in globals()
    yield "actives.csv", os.path.exists("actives.csv")
    yield "library.csv", os.path.exists("library.csv")

missing = []
for name, ok in _state():
    print(f"  {'OK  ' if ok else 'MISS'}  {name}")
    if not ok:
        missing.append(name)

print(f"\ncwd: {os.getcwd()}")
if missing:
    print(f"\nMissing: {missing}")
    print("Run the cells above in order. Nothing below will work until all are OK.")
else:
    print("\nAll present. Safe to run the preflight.")

## 5. Preflight - measure before you commit hours

Three docks. Prints the real per-compound cost on your hardware and
extrapolates. Every number in the cells below is a multiple of this one.

If a single dock takes more than 60 seconds, switch to a runtime with more
CPUs or drop EXHAUST to 4 before going further.

In [ ]:
# ---- preflight: time one dock before committing to twenty or three hundred --
import time

samples = [("known active", pd.read_csv("actives.csv").nlargest(1, "pchembl").smiles.iloc[0]),
           ("library cpd",  pd.read_csv("library.csv").smiles.iloc[0]),
           ("tranylcypromine", "NC1CC1c1ccccc1")]

times = []
for label, smi in samples:
    t = time.time()
    pq = to_pdbqt(smi)
    t_prep = time.time() - t
    if pq is None:
        print(f"  {label:<16} PREP FAILED after {t_prep:.1f}s", flush=True)
        continue
    t = time.time()
    a = dock_one(pq)
    t_dock = time.time() - t
    times.append(t_prep + t_dock)
    print(f"  {label:<16} prep {t_prep:5.1f}s  dock {t_dock:5.1f}s  -> "
          f"{'TIMEOUT' if a is None else a}", flush=True)

if not times:
    print("\n  Nothing docked. Run the self-check cell above.")
else:
    per = sum(times) / len(times)
    print(f"\n~{per:.0f}s per compound on {NCPU} CPUs")
    print(f"  validation (20 docks) -> ~{per*20/60:.0f} min")
    print(f"  {N_DOCK} compounds        -> ~{per*N_DOCK/60:.0f} min")
    print(f"  300 compounds        -> ~{per*300/60:.0f} min")
    print(f"  3000 compounds       -> ~{per*3000/3600:.1f} hours")
    if per > 60:
        print("\n  Slow. Runtime -> Change runtime type -> more CPUs, "
              "or drop EXHAUST to 4.")

## 6. Validation - do not skip

Ten known potent inhibitors versus ten random library compounds, printing as it
goes. Actives should score roughly 1-2 kcal/mol better.

This is 20 docks. Multiply the preflight number by 20 to know the wait before
you start it.

In [ ]:
# ---- VALIDATE BEFORE SCREENING ---------------------------------------------
# Dock known potent inhibitors and random library compounds. If the knowns
# don't score better, the box or the receptor is wrong and every number below
# is noise.
#
# Cost: N_VALIDATE * 2 docks. At the per-compound time the preflight printed,
# that is your wait. Prints per compound so you can see it working.
N_VALIDATE = 10          # per group. 10 is enough to see a real separation.

act = pd.read_csv("actives.csv")

# Tranylcypromine-class inhibitors bind COVALENTLY to FAD. Vina scores
# non-covalent binding only, so it cannot rank them correctly no matter how
# good the box is. Validate on reversible chemistry.
_warhead = Chem.MolFromSmarts("C1CC1N")
_is_cov = act.smiles.map(lambda s: (m := Chem.MolFromSmiles(s)) is not None
                         and m.HasSubstructMatch(_warhead))
rev = act[~_is_cov]
print(f"actives: {len(act)} total, {int(_is_cov.sum())} covalent (excluded), "
      f"{len(rev)} reversible\n")
known = rev.nlargest(N_VALIDATE, "pchembl").smiles.tolist()
decoy = pd.read_csv("library.csv").sample(N_VALIDATE, random_state=0).smiles.tolist()

def dock_set(smis, label):
    out = []
    for i, s in enumerate(smis):
        pq = to_pdbqt(s)
        if pq is None:
            print(f"  {label:<14} {i+1:>2}/{len(smis)}  prep failed", flush=True)
            continue
        t = time.time()
        a = dock_one(pq)
        print(f"  {label:<14} {i+1:>2}/{len(smis)}  "
              f"{'TIMEOUT' if a is None else f'{a:7.2f}'}  ({time.time()-t:.0f}s)",
              flush=True)
        if a is not None:
            out.append(a)
    return np.array(out)

t0 = time.time()
A = dock_set(known, "known active")
D = dock_set(decoy, "random lib")

if len(A) < 3 or len(D) < 3:
    print("\n  Too few successful docks to judge. Check the receptor cell output.")
else:
    from scipy.stats import mannwhitneyu
    sep = D.mean() - A.mean()
    p = mannwhitneyu(A, D, alternative="less").pvalue
    print(f"\n  known actives  n={len(A):2d}  mean {A.mean():6.2f}  best {A.min():6.2f}")
    print(f"  random library n={len(D):2d}  mean {D.mean():6.2f}  best {D.min():6.2f}")
    print(f"\n  separation {sep:+.2f} kcal/mol   p={p:.3g}   "
          f"({(time.time()-t0)/60:.1f} min)")
    if sep < 0.8:
        print("\n  FAIL: actives are not separating. Do not screen.")
        print("  Try the next PDB in PDB_CANDIDATES, or a larger BOX.")
    else:
        print("\n  PASS: setup discriminates. Proceed to the pose check.")

        # Calibrate the shortlist cutoff against the decoys you just measured,
        # not against a literature constant. -8.0 is meaningless if random
        # compounds average -8.7 on this receptor.
        AFFINITY_CUTOFF = round(D.mean() - 1.5 * D.std(ddof=1), 2)
        n_act = int((A <= AFFINITY_CUTOFF).sum())
        print(f"\n  calibrated AFFINITY_CUTOFF = {AFFINITY_CUTOFF} "
              f"(decoy mean - 1.5 SD)")
        print(f"  keeps {n_act}/{len(A)} known actives, rejects ~93% of random "
              f"chemistry")
        from scipy.stats import norm
        d = (D.mean() - A.mean()) / np.sqrt((A.var(ddof=1) + D.var(ddof=1)) / 2)
        print(f"  Cohen d = {d:.2f}  "
              f"({'large' if d > 0.8 else 'moderate' if d > 0.5 else 'small'})")

## 7. Pose location check

Validation says whether actives *score* better. This says whether the pose is in
the *right place*, which a score alone cannot tell you. One dock, 30 seconds.

If the pose lands more than 8 A from where the crystallographer found the bound
inhibitor, the box is misplaced no matter how good the affinities look.

In [ ]:
# ---- gold standard: can we reproduce the crystal pose? ----------------------
# The bound inhibitor was removed from the receptor. Redock it. If the setup is
# sound, it should score well AND land back roughly where the crystallographer
# found it. This is the strongest single check available, and it costs one dock.
from rdkit import Chem

xtal = np.array(list(ligs[LIG][0].values()))
print(f"crystal ligand {LIG}: {len(xtal)} atoms, centroid {xtal.mean(0).round(2)}")

# rebuild the ligand from its coordinates is unreliable without chemistry, so
# instead dock the most potent reversible active and check WHERE it lands.
probe = rev.nlargest(1, "pchembl").smiles.iloc[0]
pq = to_pdbqt(probe)
with tempfile.TemporaryDirectory() as d:
    lig_f, out_f = f"{d}/l.pdbqt", f"{d}/o.pdbqt"
    open(lig_f, "w").write(pq)
    subprocess.run(["vina", "--receptor", "receptor.pdbqt", "--ligand", lig_f,
                    "--out", out_f,
                    "--center_x", f"{centre[0]:.3f}", "--center_y", f"{centre[1]:.3f}",
                    "--center_z", f"{centre[2]:.3f}",
                    "--size_x", str(BOX), "--size_y", str(BOX), "--size_z", str(BOX),
                    "--exhaustiveness", "16", "--cpu", str(NCPU), "--seed", "42"],
                   capture_output=True, text=True)
    pose = [l for l in open(out_f) if l.startswith(("ATOM", "HETATM"))]

pos = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])] for l in pose])
d_centroid = np.linalg.norm(pos.mean(0) - xtal.mean(0))
print(f"docked pose centroid is {d_centroid:.1f} A from the crystal ligand centroid")
if d_centroid < 4:
    print("  GOOD: the pose occupies the known binding site.")
elif d_centroid < 8:
    print("  MARGINAL: same general region, different sub-pocket.")
else:
    print("  BAD: the pose is nowhere near the known site. Box is misplaced.")

## 8. Screen

Prints a line per compound, so a stall is visible immediately and you can see
whether it stalled in prep or in docking. Checkpoints every 10 results and
resumes from `docked.csv` if you rerun, so an interrupted run costs nothing.
`TIME_BUDGET_MIN` stops it cleanly rather than letting Colab drop you mid-run.

Start with `N_DOCK = 20`. Confirm it finishes, then scale using the preflight
number.

In [ ]:
# ---- screen ----------------------------------------------------------------
lib = pd.read_csv("library.csv")
if "p_active" in lib.columns:                 # triage with the QSAR model if present
    lib = lib.sort_values("p_active", ascending=False)
lib = lib.head(N_DOCK).reset_index(drop=True)

# resume: skip anything already in docked.csv
# A stale docked.csv from a previous receptor silently resumes and hands you the
# old run's numbers. That happened twice. Now it stamps the receptor and refuses
# to resume across a change.
import hashlib
rec_hash = hashlib.md5(open("receptor.pdbqt", "rb").read()).hexdigest()[:12]
stamp_ok = (os.path.exists("receptor.stamp") and
            open("receptor.stamp").read().strip() == rec_hash)

if os.path.exists("docked.csv") and not stamp_ok:
    raise SystemExit(
        "docked.csv was written against a DIFFERENT receptor. Its numbers are "
        "invalid.\n  Run:  !rm -f docked.csv docking_shortlist.csv\n"
        "  then rerun this cell.")
open("receptor.stamp", "w").write(rec_hash)

done = set()
if os.path.exists("docked.csv"):
    prev = pd.read_csv("docked.csv")
    done = set(prev.catalog_id)
    rows = prev.to_dict("records")
    print(f"resuming: {len(done):,} already docked")
else:
    rows = []

print(f"docking {len(lib):,} compounds, exhaustiveness={EXHAUST}, "
      f"budget {TIME_BUDGET_MIN} min\n")

t0 = time.time()
for i, r in lib.iterrows():
    if r.catalog_id in done:
        continue
    if (time.time() - t0) / 60 > TIME_BUDGET_MIN:
        print(f"\ntime budget reached, stopping cleanly")
        break

    t1 = time.time()
    pq = to_pdbqt(r.smiles)
    if pq is None:
        print(f"  {i+1:>5}/{len(lib)}  {r.catalog_id:<16} PREP FAILED", flush=True)
        continue
    a = dock_one(pq)
    dt = time.time() - t1
    eta = (time.time() - t0) / (i + 1) * (len(lib) - i - 1) / 60
    print(f"  {i+1:>5}/{len(lib)}  {r.catalog_id:<16} "
          f"{'TIMEOUT' if a is None else f'{a:7.2f}'}  {dt:5.0f}s  eta {eta:4.0f}m",
          flush=True)
    if a is not None:
        rows.append({"smiles": r.smiles, "catalog_id": r.catalog_id, "affinity": a})
    if len(rows) % 10 == 0 and rows:
        pd.DataFrame(rows).to_csv("docked.csv", index=False)

dock = pd.DataFrame(rows).sort_values("affinity").reset_index(drop=True)
dock.to_csv("docked.csv", index=False)
print(f"\ndocked {len(dock):,}   best {dock.affinity.min():.2f}   "
      f"median {dock.affinity.median():.2f} kcal/mol")

## 9. Shortlist

Two filters do the work:

- **Ligand efficiency** >= 0.30. Without it your top hits are just the biggest
  molecules in the library.
- **One compound per scaffold**, so a single bad assumption doesn't take out all
  fifty picks.

`max_tanimoto` is reported but not filtered on - a hit far from known actives is
exactly what this notebook exists to find, and it's also the hit most likely to
be a docking artefact. Which is what the next cell is for.

In [ ]:
# ---- shortlist -------------------------------------------------------------
# Vina rewards size: a floppy 45-heavy-atom molecule scores well without binding
# anything. Ligand efficiency (-affinity / heavy atoms) is the correction.
dms = [Chem.MolFromSmiles(s) for s in dock.smiles]
ok = [i for i, m in enumerate(dms) if m]
dock, dms = dock.iloc[ok].reset_index(drop=True), [dms[i] for i in ok]

act_fp = [FPGEN.GetFingerprint(Chem.MolFromSmiles(s))
          for s, p in zip(act.smiles, act.pchembl)
          if p >= 8.5 and Chem.MolFromSmiles(s)]
dfp = [FPGEN.GetFingerprint(m) for m in dms]

dock["max_tanimoto"] = [round(max(DataStructs.BulkTanimotoSimilarity(f, act_fp)), 3) for f in dfp]
dock["scaffold"] = [MurckoScaffold.MurckoScaffoldSmiles(mol=m, includeChirality=False) for m in dms]
dock["heavy"] = [m.GetNumHeavyAtoms() for m in dms]
dock["mw"]    = [round(Descriptors.MolWt(m), 1) for m in dms]
dock["logp"]  = [round(Crippen.MolLogP(m), 2) for m in dms]
dock["LE"]    = (-dock.affinity / dock.heavy).round(3)
dock["consensus"] = (dock.affinity.rank() + (-dock.LE).rank()) / 2

if AFFINITY_CUTOFF is None:
    raise SystemExit("AFFINITY_CUTOFF not set - run the validation cell first. "
                     "A hardcoded -8.0 passes almost everything on this receptor.")

hits = dock[(dock.LE >= 0.30) & (dock.affinity <= AFFINITY_CUTOFF)]
short = hits.sort_values("consensus").drop_duplicates("scaffold").head(50)
short.to_csv("docking_shortlist.csv", index=False)

print(f"affinity <= {AFFINITY_CUTOFF} and LE >= 0.30: {len(hits):,} of {len(dock):,}")
print(f"shortlist (one per scaffold): {len(short)}")
print(f"novel chemistry: {(short.max_tanimoto < 0.4).sum()} of {len(short)} "
      f"below 0.4 Tanimoto to known actives\n")
print(short.head(15)[["catalog_id","affinity","LE","max_tanimoto","mw","logp"]].to_string(index=False))

## 10. Refine the survivors

Vina is stochastic and the screen ran at EXHAUST=4. Re-dock the shortlist at 32
with three seeds; keep only compounds whose score is stable across seeds and
still clears the calibrated cutoff at its worst. A compound that swings by more
than 0.5 kcal/mol between seeds was a search artefact.

In [ ]:
# ---- refine the survivors ---------------------------------------------------
# The screen ran at EXHAUST=4, which is triage resolution. Vina is stochastic:
# a compound that scores well once may not score well again. Re-dock the
# shortlist at high exhaustiveness with three seeds and keep only what is
# stable. Poses that move between seeds are search artefacts.
REFINE_EXHAUST, SEEDS = 32, [1, 2, 3]

def dock_seeded(pdbqt, seed):
    with tempfile.TemporaryDirectory() as d:
        lig = f"{d}/l.pdbqt"
        open(lig, "w").write(pdbqt)
        r = subprocess.run(
            ["vina", "--receptor", "receptor.pdbqt", "--ligand", lig,
             "--out", f"{d}/o.pdbqt",
             "--center_x", f"{centre[0]:.3f}", "--center_y", f"{centre[1]:.3f}",
             "--center_z", f"{centre[2]:.3f}",
             "--size_x", str(BOX), "--size_y", str(BOX), "--size_z", str(BOX),
             "--exhaustiveness", str(REFINE_EXHAUST), "--cpu", str(NCPU),
             "--seed", str(seed)],
            capture_output=True, text=True, timeout=900)
    m = re.search(r"^\s*1\s+(-?\d+\.\d+)", r.stdout, re.M)
    return float(m.group(1)) if m else None

rows = []
for i, r in short.iterrows():
    pq = to_pdbqt(r.smiles)
    if pq is None:
        continue
    scores = [dock_seeded(pq, s) for s in SEEDS]
    scores = [x for x in scores if x is not None]
    if not scores:
        continue
    rows.append({"catalog_id": r.catalog_id, "smiles": r.smiles,
                 "triage": r.affinity, "mean": round(np.mean(scores), 2),
                 "sd": round(np.std(scores), 2), "worst": round(max(scores), 2),
                 "LE": r.LE, "max_tanimoto": r.max_tanimoto})
    print(f"  {r.catalog_id:<16} {np.mean(scores):7.2f} +/- {np.std(scores):.2f}",
          flush=True)

ref = pd.DataFrame(rows)
# stable = low seed-to-seed spread AND still beats the calibrated cutoff
ref["stable"] = (ref.sd <= 0.5) & (ref["worst"] <= AFFINITY_CUTOFF)
ref = ref.sort_values("mean")
ref.to_csv("refined_shortlist.csv", index=False)

print(f"\n{int(ref.stable.sum())} of {len(ref)} survive refinement")
print(ref[ref.stable].to_string(index=False))
print("\nOrder from the stable set only. Inspect their poses before buying.")

## 11. Inspect the poses

The single most useful sanity check. If the top compound isn't contacting FAD or
the substrate channel residues, the score is an artefact no matter how good the
number looks. Re-docks at exhaustiveness 32 for a cleaner pose.

In [ ]:
# ---- look at the poses before believing anything ---------------------------
!pip install -q py3Dmol
import py3Dmol

def show(idx=0, exhaust=32):
    """Re-dock one shortlist compound at high exhaustiveness and view the pose."""
    row = short.iloc[idx]
    pq = to_pdbqt(row.smiles)
    with tempfile.TemporaryDirectory() as d:
        lig, out = f"{d}/l.pdbqt", f"{d}/o.pdbqt"
        open(lig, "w").write(pq)
        subprocess.run(["vina", "--receptor", "receptor.pdbqt", "--ligand", lig, "--out", out,
                        "--center_x", f"{centre[0]:.3f}", "--center_y", f"{centre[1]:.3f}",
                        "--center_z", f"{centre[2]:.3f}",
                        "--size_x", str(BOX), "--size_y", str(BOX), "--size_z", str(BOX),
                        "--exhaustiveness", str(exhaust), "--cpu", str(NCPU)],
                       capture_output=True, text=True)
        pose = open(out).read()

    v = py3Dmol.view(width=760, height=520)
    v.addModel(open("receptor.pdb").read(), "pdb")
    v.setStyle({"cartoon": {"color": "lightgrey"}})
    v.addModel(pose, "pdbqt")
    v.setStyle({"model": 1}, {"stick": {"colorscheme": "cyanCarbon"}})
    v.zoomTo({"model": 1})
    print(f"{row.catalog_id}  affinity {row.affinity}  LE {row.LE}  "
          f"Tanimoto {row.max_tanimoto}")
    return v.show()

show(0)

## Then what

Docking enrichment is typically 5-20x over random and pose prediction is more
reliable than affinity prediction, so treat this as a hypothesis list, not a
result. The honest sequence from here:

1. Eyeball the top 15 poses. Discard anything not making contact.
2. Re-dock the survivors at exhaustiveness 32 with three different seeds. Keep
   what's stable across runs.
3. Order 30-50 from the survivors.
4. Submit to the free NCI-60 screen, and separately test selectivity against
   BEAS-2B or MRC-5 - NCI-60 has no normal cell lines.

Expect single-digit percent hit rate. That's what a working campaign looks
like.